In [2]:
model = "llama3.2:1b"

#### Task 1: Simple Chain with Retrieval

**Objective:**

Implement a simple RAG chain with ChatOllama, HuggingFaceEmbeddings and Chroma. 

Process: 

1. Retrieve documents from chroma db based on query
2. Invoke chain with retrieved documents as input

**Task Description:**

- load llm model via ollama
- load embedding model via ollama with `ollama pull pull bge-m3` (if not yet done)
- create chroma db client
- create prompt template for summarization
- create simple chain with following steps: retrieved documents, prompt, model, output parser
- create query and perform similarity search with a query
- invoke chain and pass retrieved documents to the chain


**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)
- [Streaming in Langchain](https://python.langchain.com/docs/concepts/streaming/)


In [3]:
from langchain_ollama import ChatOllama

# ADD HERE YOUR CODE
model = ChatOllama(model=model)

In [4]:
from langchain_ollama import OllamaEmbeddings

# ADD HERE YOUR CODE
embedding_model = OllamaEmbeddings(
    model="bge-m3",
)


In [5]:
from langchain_chroma import Chroma
import chromadb
import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    ssl=False,
    headers=None,
    settings=Settings(allow_reset=True, anonymized_telemetry=False),
    tenant=DEFAULT_TENANT,
    database=DEFAULT_DATABASE,
)

# Create a collection
# ADD HERE YOUR CODE
collection = client.get_or_create_collection("ai_model_book")


# Create chromadb
# ADD HERE YOUR CODE
vector_db_from_client = Chroma(
    client=client,
    collection_name="ai_model_book",
    embedding_function=embedding_model,
)




In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Summarize the main themes in these retrieved docs: {docs}"
)


# Convert loaded documents into strings by concatenating their content
# and ignoring metadata
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


chain = {"docs": format_docs} | prompt | model | StrOutputParser()


In [8]:
search_query = "Types of Machine Learning Systems"

# ADD HERE YOUR CODE
# Perform vector search
docs = vector_db_from_client.similarity_search(search_query)


print(docs)

[Document(metadata={'page': 33, 'source': './AI_Book.pdf'}, page_content='Types of Machine Learning Systems\nThere are so many different types of Machine Learning systems that it is useful to\nclassify them in broad categories based on:\nWhether or not they are trained with human supervision (supervised, unsuper\nvised, semisupervised, and Reinforcement Learning)\nWhether or not they can learn incrementally on the fly (online versus batch\nlearning)\nWhether they work by simply comparing new data points to known data points,\nor instead detect patterns in the training data and build a predictive model, much\nlike scientists do (instance-based versus model-based learning)\nThese criteria are not exclusive; you can combine them in any way you like. For\nexample, a state-of-the-art spam filter may learn on the fly using a deep neural net\nwork model trained using examples of spam and ham; this makes it an online, model-\nbased, supervised learning system.\nLets look at each of these crite

In [9]:
chain.invoke(docs)

'Based on the retrieved documents, here are the main themes that emerged:\n\n1. **Classification of Machine Learning Systems**: The documents discuss different types of Machine Learning systems based on their characteristics, such as:\n\t* Supervised vs. unsupervised learning\n\t* Instance-based vs. model-based learning\n\t* Online vs. batch learning\n\t* Incremental learning (e.g., online, model-based)\n2. **Supervision and Monitoring**: The importance of supervision during training is highlighted, including the need to monitor system performance and switch off or revert to a previously working state if necessary.\n3. **Generalization**: Machine Learning systems aim to generalize from their training data to new instances, with different approaches such as:\n\t* Instance-based learning: learning by heart and then generalizing\n\t* Model-based learning: building a predictive model based on patterns in the training data\n4. **Visualization and Dimensionality Reduction**: Methods for visu

In [10]:
# Simple stream the chain output
for chunk in chain.stream(docs):
    print(chunk, end="", flush=True)

The retrieved documents discuss various aspects of machine learning systems. Here are the main themes summarized:

1. **Classification**: The document highlights the different types of classification problems in machine learning, including supervised and unsupervised classification, instance-based versus model-based learning.
2. **Learning Strategies**: It explains how machine learning systems can learn from data, including batch vs. online learning, supervised vs. unsupervised learning, and reinforcement learning.
3. **Generalization**: The document emphasizes the importance of generalizing to new instances, whether through instance-based or model-based approaches.
4. **Machine Learning Techniques**: The text covers various machine learning techniques, such as:
	* Supervised/unsupervised learning
	* Instance-based versus model-based learning (e.g., PCA, LLE)
	* Association rule learning (e.g., Apriori, Eclat)
5. **Unsupervised Learning Algorithms**: It discusses visualization and dime

In [11]:
# More complex async event streaming
async for event in chain.astream_events(docs, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)


/tmp/ipykernel_1968/1672347166.py:2: LangChainBetaWarning: This API is in beta and may change in the future.
  async for event in chain.astream_events(docs, version="v2"):


Based on the retrieved documents, the main themes in Machine Learning systems are:

1. **Supervised vs Unsupervised Learning**: The difference between learning with or without human supervision (e.g., labels), and learning to generalize from new data without prior examples.

2. **Instance-Based vs Model-Based Learning**: The two primary approaches to generalization:
   - Instance-based learning: Simple, trivial form of learning that flags similar cases.
   - Model-based learning: Requires a measure of similarity between training instances and generalizes to new ones using learned patterns or models.

3. **Incremental Learning**: Online versus batch learning, which refers to how Machine Learning systems learn from data in real-time vs. learning from entire datasets at once.

4. **Generalization**: The ability of Machine Learning systems to make predictions on unseen data, and the need for these systems to perform well on new instances.

5. **Visualizations and Dimensionality Reduction**

#### Task 2: Q&A with RAG

**Objective:**

Implement a Q/A retrieval chain with ChatOllama, HuggingFaceEmbeddings and Chroma

**Task Description:**

- create RAG-Q/A prompt template
- create retriever from vector db client (instead of manually passing in docs, we automatically retrieve them from our vector store based on the user question)
- create simple chain with following steps: retriever, formatting retrieved docs, user question, prompt, model, output parser
- create question for Q/A retrieval chain
- invoke chain and with question

**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)

In [12]:

from langchain_core.runnables import RunnablePassthrough

prompt_template = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

<context>
{context}
</context>

Answer the following question:

{question}"""

# ADD HERE YOUR CODE
rag_prompt = ChatPromptTemplate.from_template(prompt_template)

# ADD HERE YOUR CODE
retriever = vector_db_from_client.as_retriever()

# ADD HERE YOUR CODE
qa_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | model
    | StrOutputParser()
)

In [13]:
qa_rag_chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x79d1ab057250>)
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], template="\nYou are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\n\n<context>\n{context}\n</context>\n\nAnswer the following question:\n\n{question}"))])
| ChatOllama(model='llama3.2:1b', _client=<ollama._client.Client object at 0x79d1c0e64b90>, _async_client=<ollama._client.AsyncClient object at 0x79d1ab9fc6d0>)
| StrOutputParser()

In [20]:
question = "What is supervised learning?"

# ADD HERE YOUR CODE
output_for_user = qa_rag_chain.invoke(question)

print(output_for_user)

Supervised learning is a type of machine learning where the training data includes labeled or annotated input features (X) and corresponding labels (y). The goal is for the algorithm to learn how to map input features to their corresponding output values, based on the observed patterns in the labeled data. This requires some form of human supervision during the training process, such as manual labeling or online data streaming.


In [21]:
# More complex async event streaming
async for event in qa_rag_chain.astream_events(question, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Supervised learning is a type of machine learning where the training data includes the desired outcomes or labels, allowing the algorithm to learn by itself what is the best strategy (policy) to get the most reward over time. The system must then be fine-tuned using supervised learning techniques after initial unsupervised or semi-supervised training.

#### Alternative: Using pre-built ConversationalRetrievalChain Class

In [16]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

In [17]:
retriever = vector_db_from_client.as_retriever()
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [18]:
qa_chain = ConversationalRetrievalChain.from_llm(
    model, retriever=retriever, memory=memory, verbose=False
)

In [19]:
# More complex async event streaming
async for event in qa_chain.astream_events("What is supervised learning?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Supervised learning is a type of machine learning where an algorithm is trained on labeled data, which means that the input data includes examples with corresponding labels or targets. The goal of supervised learning is to learn a mapping between input data and output labels, so that the algorithm can make predictions or classifications on new, unseen data.

In a typical supervised learning scenario:

* An algorithm is trained on a dataset where each sample has a label or target associated with it.
* The training dataset includes a set of examples (e.g., images, text) and corresponding labels (e.g., "cat" or "dog").
* The algorithm learns to map the input data to the correct output label through the process of learning.

Supervised learning is often contrasted with unsupervised learning, where the algorithm discovers patterns or relationships in the data without any labeled targets. Reinforcement learning is another type of machine learning that falls under the category of supervised l

In [ ]:
# More complex async event streaming
async for event in qa_chain.astream_events("Which algorithms can be used there?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)